# MTMC 11c \u2014 WILDTRACK Stages 4-5 (Association + Evaluation)

Cross-camera person association and evaluation on WILDTRACK.

**Prerequisite**: attach **11b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-11b-wildtrack-stage-3" -> add`

**This is the iteration loop** \u2014 edit the tuning params cell and re-run.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + graph clustering) | ~2 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _major, _minor = _cap.strip().split(".")
        _sm = int(_major) * 10 + int(_minor)
        if _sm < 70:
            print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) \u2014 installing compatible PyTorch ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu124",
            ])
            print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
PROJECT = Path("/kaggle/working/gp")

if not PROJECT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/people-tracking", REPO_URL, str(PROJECT)])
os.chdir(str(PROJECT))

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

try:
    import trackeval; print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("trackeval", "trackeval"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 11b

In [ ]:
DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)

def print_deep_listing(root, max_depth=3, prefix=""):
    root = Path(root)
    for item in sorted(root.iterdir()):
        print(f"{prefix}{item.name}{'/' if item.is_dir() else ''}")
        if item.is_dir() and len(prefix) < max_depth * 2:
            try:
                print_deep_listing(item, max_depth, prefix + "  ")
            except PermissionError:
                print(f"{prefix}  [permission denied]")


print("=== /kaggle/input/ deep listing ===")
print_deep_listing(Path("/kaggle/input"))

INPUT_11B = None
for ckpt_file in Path("/kaggle/input").rglob("checkpoint.tar.gz"):
    INPUT_11B = ckpt_file.parent
    print(f"Found checkpoint at: {ckpt_file}")
    break

if INPUT_11B is None:
    print("=== /kaggle/input/ deep listing ===")
    print_deep_listing(Path("/kaggle/input"))
    raise FileNotFoundError("11b output not found in /kaggle/input/")

ckpt = INPUT_11B / "checkpoint.tar.gz"
print(f"Checkpoint: {ckpt} ({ckpt.stat().st_size/1024**2:.1f} MB)")

with tarfile.open(str(ckpt), "r:gz") as tar:
    tar.extractall(path=str(DATA_OUT))

with open(DATA_OUT / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
print(f"\u2713 Run: {RUN_NAME}")

run_dir = DATA_OUT / RUN_NAME
for stage in ["stage1", "stage2", "stage3"]:
    sd = run_dir / stage
    if sd.exists():
        nf = sum(1 for _ in sd.rglob("*") if _.is_file())
        print(f"  {stage}: {nf} files")

gt_dir = DATA_OUT / "gt_annotations"
GT_DIR = str(gt_dir) if gt_dir.exists() else ""
print(f"GT: {GT_DIR or 'NOT FOUND'}")

## 3. Tuning Parameters

**Edit these values** then re-run the cells below. No need to re-run 11a or 11b.

In [ ]:
# === TUNING PARAMS (edit these and re-run) ===
SIM_THRESH        = 0.40
ALGORITHM         = "conflict_free_cc"
APPEARANCE_WEIGHT = 0.65
HSV_WEIGHT        = 0.10
LOUVAIN_RES       = 1.0
AQE_K             = 5
FIC_REG           = 0.05
TEMPORAL_OVERLAP_BONUS = 0.20
TEMPORAL_OVERLAP_MAX_TIME = 5.0
INTRA_MERGE_THRESH = 0.65
INTRA_MERGE_GAP   = 60

## 4. Run Stages 4-5

In [ ]:
import subprocess, time

os.chdir(str(PROJECT))

cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/wildtrack.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
    "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    "--override", f"stage4.association.weights.person.appearance={APPEARANCE_WEIGHT}",
    "--override", f"stage4.association.weights.person.hsv={HSV_WEIGHT}",
    "--override", f"stage4.association.fic.regularisation={FIC_REG}",
    "--override", f"stage4.association.query_expansion.k={AQE_K}",
    "--override", f"stage4.association.temporal_overlap.bonus={TEMPORAL_OVERLAP_BONUS}",
    "--override", f"stage4.association.temporal_overlap.max_mean_time={TEMPORAL_OVERLAP_MAX_TIME}",
    "--override", "stage4.association.temporal_overlap.min_ratio=0.15",
    "--override", "stage4.association.reranking.enabled=false",
    "--override", "stage4.association.query_expansion.alpha=5.0",
    "--override", "stage4.association.weights.person.spatiotemporal=0.25",
    "--override", "stage4.association.exhaustive_min_similarity=0.20",
    "--override", "stage4.association.mutual_nn.top_k_per_query=8",
    "--override", "stage4.association.gallery_expansion.threshold=0.40",
    "--override", "stage4.association.graph.max_component_size=20",
    "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
    "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
]

print("Running:")
print(" ".join(cmd))
t0 = time.time()
result = subprocess.run(cmd, cwd=str(PROJECT))
print(f"Done with return code {result.returncode} in {(time.time()-t0)/60:.1f} min")
if result.returncode != 0:
    raise RuntimeError("Stages 4-5 failed")

## 5. Results

In [ ]:
report_path = DATA_OUT / RUN_NAME / "stage5" / "evaluation_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)

    m = report.get("metrics", report)
    print("=" * 60)
    print("WILDTRACK PERSON PIPELINE RESULTS")
    print("=" * 60)
    print(f"  IDF1:      {m.get('idf1', m.get('IDF1', 0)):.4f}")
    print(f"  MTMC_IDF1: {m.get('mtmc_idf1', 0):.4f}")
    print(f"  MOTA:      {m.get('MOTA', m.get('mota', 0)):.4f}")
    print(f"  HOTA:      {m.get('HOTA', m.get('hota', 0)):.4f}")

    details = report.get("details", {})
    if details:
        print(f"\n  MTMC_MOTA: {details.get('mtmc_mota', 'N/A')}")
        print(f"  ID_sw:     {details.get('id_switches', 'N/A')}")
        print(f"  Tracklets: {details.get('num_tracklets', 'N/A')}")

    # Per-camera breakdown
    per_cam = report.get("per_camera", {})
    if per_cam:
        print(f"\nPer-camera IDF1:")
        for cam, cm in sorted(per_cam.items()):
            idf1 = cm.get("idf1", cm.get("IDF1", 0))
            print(f"  {cam}: {idf1:.3f}")
else:
    print(f"No evaluation report found at {report_path}")
    # Check if stage4/5 output exists
    for sn in ["stage4", "stage5"]:
        sd = DATA_OUT / RUN_NAME / sn
        if sd.exists():
            print(f"  {sn} output exists ({sum(1 for _ in sd.rglob('*'))} files)")